# Capstone Data Processing
**Project:** Predicting Long-Run Home Price Appreciation Across U.S. Housing Markets  
**Notebook:** 02: Data Processing & Panel Construction  

This notebook:
1. Loads raw data saved by Notebook 01
2. Converts quarterly HPI to annual and computes year-over-year appreciation rates
3. Computes population growth rate from ACS population data
4. Merges FHFA, Census ACS, and FRED into a single clean panel dataset
5. Handles the 2020 data gap and other missing value issues
6. Saves the processed panel to `data/processed/panel_dataset.csv`

## Setup

In [2]:
import pandas as pd
import numpy as np
import os

os.makedirs('data/processed', exist_ok=True)

print('Setup complete.')

Setup complete.


---
## 1. Load Raw Data

In [3]:
hpi  = pd.read_csv('data/raw/hpi_po_metro.csv', dtype={'cbsa': str})
acs  = pd.read_csv('data/raw/acs_indicators.csv', dtype={'cbsa': str})
mort = pd.read_csv('data/raw/fred_mortgage30us.csv')

# Ensure CBSA codes are zero-padded 5-digit strings
hpi['cbsa']  = hpi['cbsa'].str.zfill(5)
acs['cbsa']  = acs['cbsa'].str.zfill(5)

print('HPI shape: ',  hpi.shape,  ' | years:', hpi['yr'].min(), '–', hpi['yr'].max())
print('ACS shape: ',  acs.shape,  ' | years:', acs['year'].min(), '–', acs['year'].max())
print('Mort shape: ', mort.shape, ' | years:', mort['yr'].min(), '–', mort['yr'].max())
print()
print('HPI columns:', hpi.columns.tolist())
print('ACS columns:', acs.columns.tolist())
print('Mort columns:', mort.columns.tolist())

HPI shape:  (14000, 6)  | years: 1991 – 2025
ACS shape:  (6772, 10)  | years: 2011 – 2024
Mort shape:  (66, 3)  | years: 2010 – 2026

HPI columns: ['cbsa', 'metro_name', 'yr', 'qtr', 'index_nsa', 'index_sa']
ACS columns: ['cbsa', 'metro_name_acs', 'year', 'population', 'unemployment_rate', 'poverty_rate', 'median_income', 'pct_bachelors_plus', 'vacancy_rate', 'homeownership_rate']
Mort columns: ['yr', 'qtr', 'mortgage30us']


---
## 2. Convert Quarterly HPI to Annual and Compute Appreciation Rate

I use the **Q4 seasonally-adjusted index value** as each year's representative price level.  
Year-over-year appreciation rate = ((HPI_t − HPI_{t−1}) / HPI_{t−1}) × 100

In [4]:
# Extract Q4 (end-of-year) observations
hpi_q4 = hpi[hpi['qtr'] == 4][['cbsa', 'metro_name', 'yr', 'index_sa']].copy()
hpi_q4 = hpi_q4.rename(columns={'yr': 'year', 'index_sa': 'hpi'})
hpi_q4 = hpi_q4.sort_values(['cbsa', 'year']).reset_index(drop=True)

# Compute year-over-year appreciation rate within each MSA
hpi_q4['appreciation_rate'] = (
    hpi_q4.groupby('cbsa')['hpi']
    .pct_change() * 100
)

# Drop rows where appreciation cannot be computed (first year per MSA)
hpi_annual = hpi_q4.dropna(subset=['appreciation_rate']).copy()

print(f'Annual HPI rows (after computing appreciation): {len(hpi_annual):,}')
print(f'Years: {hpi_annual["year"].min()} – {hpi_annual["year"].max()}')
print(f'Unique MSAs: {hpi_annual["cbsa"].nunique()}')
print()
print('Appreciation rate summary statistics:')
print(hpi_annual['appreciation_rate'].describe().round(2))
print()
# Preview: top and bottom appreciating MSA-years
print('Top 5 appreciation observations:')
print(hpi_annual.nlargest(5, 'appreciation_rate')[['metro_name','year','appreciation_rate']].to_string(index=False))
print()
print('Bottom 5 appreciation observations:')
print(hpi_annual.nsmallest(5, 'appreciation_rate')[['metro_name','year','appreciation_rate']].to_string(index=False))

Annual HPI rows (after computing appreciation): 3,400
Years: 1992 – 2025
Unique MSAs: 100

Appreciation rate summary statistics:
count    3400.00
mean        4.70
std         7.21
min       -38.23
25%         1.35
50%         4.47
75%         8.19
max        38.93
Name: appreciation_rate, dtype: float64

Top 5 appreciation observations:
                             metro_name  year  appreciation_rate
              Phoenix-Mesa-Chandler, AZ  2005          38.929889
Las Vegas-Henderson-North Las Vegas, NV  2004          38.166069
              Cape Coral-Fort Myers, FL  2021          34.920842
          Orlando-Kissimmee-Sanford, FL  2005          33.343178
              Cape Coral-Fort Myers, FL  2005          32.254898

Bottom 5 appreciation observations:
                             metro_name  year  appreciation_rate
              Cape Coral-Fort Myers, FL  2008         -38.228399
Las Vegas-Henderson-North Las Vegas, NV  2008         -36.791555
                      Stockton-Lodi, CA

---
## 3. Compute Population Growth Rate from ACS Data

Population growth rate = ((population_t − population_{t−1}) / population_{t−1}) × 100  
Note: 2020 is missing from ACS; growth rate for 2021 will be computed relative to 2019.

In [ ]:
acs_sorted = acs.sort_values(['cbsa', 'year']).copy()

# Population growth rate — year-over-year within each MSA
acs_sorted['pop_growth_rate'] = (
    acs_sorted.groupby('cbsa')['population']
    .pct_change() * 100
)

# Flag the 2021 observations where growth rate spans 2 years (2019-2021)
# due to the missing 2020 ACS. Divide by 2 to approximate annual rate.
acs_sorted['pop_growth_rate'] = np.where(
    acs_sorted['year'] == 2021,
    acs_sorted['pop_growth_rate'] / 2,
    acs_sorted['pop_growth_rate']
)

print('ACS predictors after adding population growth rate:')
print(acs_sorted[['cbsa','year','population','pop_growth_rate',
                   'unemployment_rate','poverty_rate','median_income',
                   'pct_bachelors_plus','vacancy_rate','homeownership_rate']].describe().round(2))

ACS predictors after adding population growth rate:
          year   population  pop_growth_rate  unemployment_rate  poverty_rate  \
count  6772.00      6772.00          6178.00            6772.00       6771.00   
mean   2017.31    564026.01             0.68               6.42         15.66   
std       4.14   1432854.53             4.97               3.20          6.22   
min    2011.00     54080.00           -70.46               0.85          3.80   
25%    2014.00    100294.50            -0.22               4.21         11.78   
50%    2017.00    166075.50             0.42               5.69         14.67   
75%    2021.00    414164.25             1.20               7.90         18.16   
max    2024.00  20320876.00           203.73              36.33         62.71   

       median_income  pct_bachelors_plus  vacancy_rate  homeownership_rate  
count        6772.00             6752.00       6772.00             6772.00  
mean        56212.24               27.21         12.88          

---
## 4. Compute Annual Average Mortgage Rate from FRED Quarterly Data

In [6]:
# Average the four quarterly observations per year
mort_annual = (
    mort.groupby('yr')['mortgage30us']
    .mean()
    .reset_index()
    .rename(columns={'yr': 'year', 'mortgage30us': 'mortgage_rate'})
)

print('Annual mortgage rate:')
print(mort_annual.to_string(index=False))

Annual mortgage rate:
 year  mortgage_rate
 2010         4.7025
 2011         4.4475
 2012         3.6550
 2013         3.9750
 2014         4.1725
 2015         3.8475
 2016         3.6550
 2017         3.9875
 2018         4.5425
 2019         3.9350
 2020         3.1175
 2021         2.9575
 2022         5.3425
 2023         6.8050
 2024         6.7225
 2025         6.6050
 2026         6.2550


### 4.1 Crosswalk Mapping CBSA Codes 
Map Metropolitan CBSA codes (FHFA) to parent MSA CBSA codes (ACS) before merging. For 34 markets, the FHFA indexes price data at the division level while the Census ACS publishes demographic data at the parent MSA level. Cleveland's additional mismatch (FHFA: 17410, ACS: 17460) is included here alongside the 33 MSADs.

In [ ]:
# Crosswalk mapping MSAD CBSA codes (FHFA) to parent MSA CBSA codes (ACS)
# These 33 MSADs have HPI data in FHFA but only appear in ACS 
# at their parent MSA level
msad_to_msa = {
    '11244': '31080',  # Anaheim -> Los Angeles-Long Beach-Anaheim
    '11694': '47900',  # Arlington-Alexandria -> Washington-Arlington-Alexandria
    '12054': '12060',  # Atlanta MSAD -> Atlanta-Sandy Springs-Roswell
    '14454': '14460',  # Boston MSAD -> Boston-Cambridge-Newton
    '15764': '14460',  # Cambridge-Newton -> Boston-Cambridge-Newton
    '15804': '37980',  # Camden -> Philadelphia-Camden-Wilmington
    '16984': '16980',  # Chicago-Naperville -> Chicago-Naperville-Elgin
    '17410': '17460',  # Cleveland (FHFA) -> Cleveland-Elyria, OH (ACS) *discovered upon code mismatch
    '19124': '19100',  # Dallas-Plano -> Dallas-Fort Worth-Arlington
    '19804': '19820',  # Detroit-Dearborn -> Detroit-Warren-Dearborn
    '20994': '16980',  # Elgin -> Chicago-Naperville-Elgin
    '21794': '42660',  # Everett -> Seattle-Tacoma-Bellevue
    '22744': '33100',  # Fort Lauderdale -> Miami-Fort Lauderdale-Pompano Beach
    '23104': '19100',  # Fort Worth-Arlington -> Dallas-Fort Worth-Arlington
    '23224': '47900',  # Frederick-Gaithersburg -> Washington-Arlington-Alexandria
    '29484': '35620',  # Lakewood-New Brunswick -> New York-Newark-Jersey City
    '31084': '31080',  # Los Angeles-Long Beach -> Los Angeles-Long Beach-Anaheim
    '31924': '12060',  # Marietta -> Atlanta-Sandy Springs-Roswell
    '33124': '33100',  # Miami-Miami Beach -> Miami-Fort Lauderdale-Pompano Beach
    '33874': '37980',  # Montgomery County-Bucks -> Philadelphia-Camden-Wilmington
    '35004': '35620',  # Nassau-Suffolk -> New York-Newark-Jersey City
    '35614': '35620',  # New York-Jersey City -> New York-Newark-Jersey City
    '35084': '35620',  # Newark -> New York-Newark-Jersey City
    '36084': '41860',  # Oakland-Fremont -> San Francisco-Oakland-Hayward
    '37964': '37980',  # Philadelphia -> Philadelphia-Camden-Wilmington
    '41884': '41860',  # San Francisco-San Mateo -> San Francisco-Oakland-Hayward
    '42644': '42660',  # Seattle-Bellevue -> Seattle-Tacoma-Bellevue
    '41304': '45300',  # St. Petersburg -> Tampa-St. Petersburg-Clearwater
    '45104': '42660',  # Tacoma-Lakewood -> Seattle-Tacoma-Bellevue
    '45294': '45300',  # Tampa -> Tampa-St. Petersburg-Clearwater
    '47664': '19820',  # Warren-Troy -> Detroit-Warren-Dearborn
    '47764': '47900',  # Washington DC-MD -> Washington-Arlington-Alexandria
    '48424': '33100',  # West Palm Beach -> Miami-Fort Lauderdale-Pompano Beach
    '48864': '37980',  # Wilmington -> Philadelphia-Camden-Wilmington
}

# Apply MSAD crosswalk before merging
# Maps Metropolitan Division CBSA codes (FHFA) to parent MSA codes (ACS)
# so that demographic predictors can be matched to division-level HPI data
hpi_annual['cbsa_for_merge'] = hpi_annual['cbsa'].replace(msad_to_msa)

print(f"MSAs before crosswalk: {hpi_annual['cbsa'].nunique()}")
print(f"Unique merge keys after crosswalk: {hpi_annual['cbsa_for_merge'].nunique()}")
print(f"MSADs mapped to parent MSAs: {hpi_annual['cbsa'].ne(hpi_annual['cbsa_for_merge']).sum()} rows")

MSAs before crosswalk: 100
Unique merge keys after crosswalk: 80
MSADs mapped to parent MSAs: 1156 rows


---
## 5. Merge All Sources into a Panel Dataset

Merge order:
1. HPI annual appreciation (100 MSAs × years)
2. LEFT JOIN ACS indicators on (cbsa, year)
3. LEFT JOIN mortgage rate on (year)

In [8]:
# Merge HPI with ACS
panel = hpi_annual.merge(
    acs_sorted[['cbsa', 'year', 'pop_growth_rate', 'unemployment_rate',
                'poverty_rate', 'median_income', 'pct_bachelors_plus',
                'vacancy_rate', 'homeownership_rate']],
    left_on=['cbsa_for_merge', 'year'],
    right_on=['cbsa', 'year'],
    how='left'
).drop(columns=['cbsa_for_merge', 'cbsa_y']).rename(columns={'cbsa_x': 'cbsa'})

# Merge with annual mortgage rate
panel = panel.merge(mort_annual, on='year', how='left')

print(f'Panel shape after merge: {panel.shape}')
print(f'Years covered: {panel["year"].min()} – {panel["year"].max()}')
print(f'Unique MSAs: {panel["cbsa"].nunique()}')
print()
print('Missing values per column:')
print(panel.isnull().sum())



Panel shape after merge: (3400, 13)
Years covered: 1992 – 2025
Unique MSAs: 100

Missing values per column:
cbsa                     0
metro_name               0
year                     0
hpi                      0
appreciation_rate        0
pop_growth_rate       2214
unemployment_rate     2114
poverty_rate          2114
median_income         2114
pct_bachelors_plus    2114
vacancy_rate          2114
homeownership_rate    2114
mortgage_rate         1800
dtype: int64


In [9]:
# Restrict to years where ACS data is available (2012 onward, since
# pop_growth_rate requires two consecutive ACS years, first being 2012)
panel = panel[panel['year'] >= 2012].copy()

# Drop rows missing any predictor or outcome variable
predictor_cols = [
    'pop_growth_rate', 'unemployment_rate', 'poverty_rate',
    'median_income', 'pct_bachelors_plus', 'vacancy_rate',
    'homeownership_rate', 'mortgage_rate'
]
before = len(panel)
panel = panel.dropna(subset=['appreciation_rate'] + predictor_cols)
after  = len(panel)

print(f'Rows before dropping missing: {before:,}')
print(f'Rows after dropping missing:  {after:,}')
print(f'Rows dropped: {before - after:,}')
print()
print(f'Final panel: {panel["cbsa"].nunique()} MSAs × {panel["year"].nunique()} years')
print(f'Year range: {panel["year"].min()} – {panel["year"].max()}')
print()
print('Final column list:')
print(panel.columns.tolist())

Rows before dropping missing: 1,400
Rows after dropping missing:  1,186
Rows dropped: 214

Final panel: 100 MSAs × 12 years
Year range: 2012 – 2024

Final column list:
['cbsa', 'metro_name', 'year', 'hpi', 'appreciation_rate', 'pop_growth_rate', 'unemployment_rate', 'poverty_rate', 'median_income', 'pct_bachelors_plus', 'vacancy_rate', 'homeownership_rate', 'mortgage_rate']


### 5.1 Merge Verification
Confirm all 100 FHFA MSAs are present after applying the crosswalk.

In [10]:
# Verify all 100 FHFA MSAs made it into the final panel
# Should return 0 after MSAD crosswalk is applied
fhfa_cbsas  = set(hpi_annual['cbsa'].unique())
panel_cbsas = set(panel['cbsa'].unique())
lost_cbsas  = fhfa_cbsas - panel_cbsas

lost = hpi_annual[hpi_annual['cbsa'].isin(lost_cbsas)][['cbsa','metro_name']].drop_duplicates()
print(f'{len(lost)} MSAs lost during merge:\n')
print(lost.sort_values('metro_name').to_string(index=False))

0 MSAs lost during merge:

Empty DataFrame
Columns: [cbsa, metro_name]
Index: []


### 5.2 CBSA Code Investigation
Two additional metropolitan areas had code mismatches not covered by the MSAD crosswalk. I investigated why Cleveland, OH and Dayton, OH had only 1 and 4 observations, respectively.


In [11]:
# Check what CBSA code FHFA uses for Cleveland
print('Cleveland in HPI data:')
print(hpi_annual[hpi_annual['metro_name'].str.contains('Cleveland', na=False)]
      [['cbsa', 'metro_name']].drop_duplicates().to_string(index=False))

# Check what Cleveland entries exist in ACS
print('\nCleveland entries in ACS:')
print(acs_sorted[acs_sorted['metro_name_acs'].str.contains('Cleveland', na=False)]
      [['cbsa', 'metro_name_acs']].drop_duplicates().to_string(index=False))

# Check how many panel rows Cleveland has and which year
print('\nCleveland rows in final panel:')
print(panel[panel['metro_name'].str.contains('Cleveland', na=False)]
      [['metro_name', 'year', 'cbsa']].to_string(index=False))

Cleveland in HPI data:
 cbsa    metro_name
17410 Cleveland, OH

Cleveland entries in ACS:
 cbsa                         metro_name_acs
17410               Cleveland, OH Metro Area
17420               Cleveland, TN Metro Area
17460 Cleveland-Elyria-Mentor, OH Metro Area
17460        Cleveland-Elyria, OH Metro Area

Cleveland rows in final panel:
   metro_name  year  cbsa
Cleveland, OH  2012 17410
Cleveland, OH  2013 17410
Cleveland, OH  2014 17410
Cleveland, OH  2015 17410
Cleveland, OH  2016 17410
Cleveland, OH  2017 17410
Cleveland, OH  2018 17410
Cleveland, OH  2019 17410
Cleveland, OH  2021 17410
Cleveland, OH  2022 17410


In [12]:
print('Dayton in HPI data:')
print(hpi_annual[hpi_annual['metro_name'].str.contains('Dayton', na=False)]
      [['cbsa', 'metro_name']].drop_duplicates().to_string(index=False))

print('\nDayton entries in ACS:')
print(acs_sorted[acs_sorted['metro_name_acs'].str.contains('Dayton', na=False)]
      [['cbsa', 'metro_name_acs']].drop_duplicates().to_string(index=False))

Dayton in HPI data:
 cbsa                       metro_name
19430 Dayton-Kettering-Beavercreek, OH

Dayton entries in ACS:
 cbsa                                    metro_name_acs
19380                             Dayton, OH Metro Area
19430                   Dayton-Kettering, OH Metro Area
19430       Dayton-Kettering-Beavercreek, OH Metro Area
19660 Deltona-Daytona Beach-Ormond Beach, FL Metro Area


In [13]:
print(acs_sorted[acs_sorted['cbsa'].isin(['19380', '19430'])]
      [['cbsa', 'metro_name_acs', 'year']]
      .sort_values('year').to_string(index=False))

 cbsa                              metro_name_acs  year
19380                       Dayton, OH Metro Area  2011
19380                       Dayton, OH Metro Area  2012
19380                       Dayton, OH Metro Area  2013
19380                       Dayton, OH Metro Area  2014
19380                       Dayton, OH Metro Area  2015
19380                       Dayton, OH Metro Area  2016
19380                       Dayton, OH Metro Area  2017
19380                       Dayton, OH Metro Area  2018
19430             Dayton-Kettering, OH Metro Area  2019
19430             Dayton-Kettering, OH Metro Area  2021
19430             Dayton-Kettering, OH Metro Area  2022
19430 Dayton-Kettering-Beavercreek, OH Metro Area  2023
19430 Dayton-Kettering-Beavercreek, OH Metro Area  2024


In [14]:
# Remove MSAs with too few observations to contribute meaningfully to the model
# Dayton-Kettering had only 4 observations due to ACS CBSA reclassification in 2019
min_obs = 6
obs_counts = panel.groupby('metro_name').size()
sufficient  = obs_counts[obs_counts >= min_obs].index
panel = panel[panel['metro_name'].isin(sufficient)]

print(f'MSAs removed for insufficient observations: {(obs_counts < min_obs).sum()}')
print(f'Final panel: {panel["cbsa"].nunique()} MSAs × {panel["year"].nunique()} years')
print(f'Total observations: {len(panel):,}')

MSAs removed for insufficient observations: 1
Final panel: 99 MSAs × 12 years
Total observations: 1,182


In [15]:
# Remove observations with extreme pop_growth_rate values caused by
# CBSA boundary redefinitions in the ACS (not real population change)
extreme_mask = panel['pop_growth_rate'] < -5.0
print('Removing the following CBSA boundary artifact observations:')
print(panel[extreme_mask][['metro_name', 'year', 'pop_growth_rate']].to_string(index=False))

panel = panel[~extreme_mask].copy()
print(f'\nRows removed: {extreme_mask.sum()}')
print(f'Rows remaining: {len(panel):,}')

Removing the following CBSA boundary artifact observations:
                              metro_name  year  pop_growth_rate
                          Birmingham, AL  2019        -5.327830
Hartford-West Hartford-East Hartford, CT  2023        -5.711932
                New Orleans-Metairie, LA  2023       -22.790601
                           Worcester, MA  2023       -11.619680

Rows removed: 4
Rows remaining: 1,178


In [16]:
# Visual inspection of merged panel structure
panel.sample(5)

,cbsa,metro_name,year,hpi,appreciation_rate,pop_growth_rate,unemployment_rate,poverty_rate,median_income,pct_bachelors_plus,vacancy_rate,homeownership_rate,mortgage_rate
333,12940,"Baton Rouge, LA",2019,273.34,1.005099,2.835765,5.295203,15.041841,60746.0,27.527869,17.956824,69.697912,3.9350
2915,41740,"San Diego-Chula Vista-Carlsbad, CA",2017,310.32,8.333042,0.600889,5.608083,11.807814,76207.0,38.784704,7.234958,53.492439,3.9875
1012,19780,"Des Moines-West Des Moines, IA",2018,241.50,4.324161,1.470481,3.275799,9.304284,71352.0,38.893702,7.637961,69.616357,4.5425
1251,23224,"Frederick-Gaithersburg-Bethesda, MD (MSAD)",2019,269.21,3.339603,0.471219,4.084089,7.547543,105659.0,51.401852,6.266170,63.518780,3.9350
268,12540,"Bakersfield-Delano, CA",2022,296.74,7.250253,-0.170540,7.002688,17.866757,66234.0,19.041598,7.410492,60.960813,5.3425


---
## 6. Descriptive Summary of Final Panel

In [22]:
summary_cols = ['appreciation_rate'] + predictor_cols
print('Panel dataset (descriptive statistics):')
print(panel[summary_cols].describe().round(3).to_string())

Panel dataset (descriptive statistics):
       appreciation_rate  pop_growth_rate  unemployment_rate  poverty_rate  median_income  pct_bachelors_plus  vacancy_rate  homeownership_rate  mortgage_rate
count           1178.000         1178.000           1178.000      1178.000       1178.000            1178.000      1178.000            1178.000       1178.000
mean               7.077            1.083              5.858        12.774      69860.243              35.871         9.244              63.301          4.459
std                5.172            1.989              2.061         3.145      17984.272               7.618         4.017               5.435          1.155
min               -6.882           -3.820              2.332         6.316      39873.000              14.379         2.656              47.447          2.958
25%                3.801            0.263              4.365        10.503      56277.000              30.921         6.531              60.486          3.703
50%   

In [ ]:
# Observations per MSA
obs_per_msa = panel.groupby('metro_name').size()
print('Observations per MSA:')
print(obs_per_msa.describe().round(1))
print()
print('MSAs with fewest observations (potential data gaps):')
print(obs_per_msa.nsmallest(10).to_string())

Observations per MSA:
count    99.0
mean     11.9
std       0.4
min      10.0
25%      12.0
50%      12.0
75%      12.0
max      12.0
dtype: float64

MSAs with fewest observations (potential data gaps):
metro_name
Anaheim-Santa Ana-Irvine, CA (MSAD)           10
Cleveland, OH                                 10
Los Angeles-Long Beach-Glendale, CA (MSAD)    10
Birmingham, AL                                11
Hartford-West Hartford-East Hartford, CT      11
New Orleans-Metairie, LA                      11
Worcester, MA                                 11
Albany-Schenectady-Troy, NY                   12
Albuquerque, NM                               12
Allentown-Bethlehem-Easton, PA-NJ             12


In [ ]:
# Appreciation by year
appreciation_by_year = panel.groupby('year')['appreciation_rate'].agg(['mean','median','std']).round(2)
print('Average appreciation rate by year:')
print(appreciation_by_year.to_string())

Average appreciation rate by year:
       mean  median   std
year                     
2012   5.97    4.61  5.76
2013   8.24    7.04  5.90
2014   5.05    4.73  3.17
2015   6.15    5.58  3.82
2016   6.47    6.43  2.92
2017   6.65    6.30  2.74
2018   5.51    5.45  2.60
2019   5.55    5.33  2.11
2021  17.47   16.60  5.56
2022   7.14    7.84  4.53
2023   6.29    6.34  3.35
2024   4.36    4.62  3.27


In [20]:
# Inspect extreme values before saving
print('Lowest pop_growth_rate observations:')
print(panel.nsmallest(5, 'pop_growth_rate')[['metro_name','year','pop_growth_rate']].to_string(index=False))

print('\nHighest vacancy_rate observations:')
print(panel.nlargest(5, 'vacancy_rate')[['metro_name','year','vacancy_rate']].to_string(index=False))

Lowest pop_growth_rate observations:
                                     metro_name  year  pop_growth_rate
              Lakewood-New Brunswick, NJ (MSAD)  2019        -3.820395
        Nassau County-Suffolk County, NY (MSAD)  2019        -3.820395
                              Newark, NJ (MSAD)  2019        -3.820395
New York-Jersey City-White Plains, NY-NJ (MSAD)  2019        -3.820395
             Louisville/Jefferson County, KY-IN  2013        -3.034956

Highest vacancy_rate observations:
               metro_name  year  vacancy_rate
Cape Coral-Fort Myers, FL  2013     35.148112
Cape Coral-Fort Myers, FL  2012     34.150607
Cape Coral-Fort Myers, FL  2017     32.292459
Cape Coral-Fort Myers, FL  2016     32.035595
Cape Coral-Fort Myers, FL  2015     30.434760


---
## 7. Save Processed Panel Dataset

In [21]:
panel.to_csv('data/processed/panel_dataset.csv', index=False)

print('Saved: data/processed/panel_dataset.csv')
print()
print('Final panel summary:')
print(f'  Total observations:  {len(panel):,}')
print(f'  Unique MSAs:         {panel["cbsa"].nunique()}')
print(f'  Year range:          {panel["year"].min()} – {panel["year"].max()}')
print(f'  Outcome variable:    appreciation_rate (% YoY HPI change)')
print(f'  Predictor variables: {predictor_cols}')

Saved: data/processed/panel_dataset.csv

Final panel summary:
  Total observations:  1,178
  Unique MSAs:         99
  Year range:          2012 – 2024
  Outcome variable:    appreciation_rate (% YoY HPI change)
  Predictor variables: ['pop_growth_rate', 'unemployment_rate', 'poverty_rate', 'median_income', 'pct_bachelors_plus', 'vacancy_rate', 'homeownership_rate', 'mortgage_rate']
